# P1 Evaluation: CTPGesture v1 — Metrics and Keyframe Figures

**Independent copy:** original evaluation notebooks are preserved. This new notebook supports a fresh Kaggle Save & Run All from the fine-tuned checkpoint (no training), with verified native CUDA wheels selected from an attached Kaggle Dataset. Two separate qualitative protocols: isolated clip filmstrips and continuous GT/pred timelines.

**Important:** Python 3.13 needs correctly built cp313 wheels **plus matching Torch/CUDA/ABI/SM**. Attach the wheel builder output dataset containing `wheel_manifest.json`, both `.whl` files and `verify_install.py`. If the CUDA environment changed, the notebook fails before expensive inference rather than silently reporting a different backend.


**P1 standalone execution order:** run this notebook top-to-bottom; it excludes the old single-video debug pass, postprocessing ablation, and unrelated VideoMAE/computation comparisons. It only performs checkpoint inference + the requested isolated and continuous metrics/figures.


## 0. Runtime setup

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path
import json
# Keep the same software family used by the training notebook.
packages = [
    "setuptools<82",
    "wheel",
    "packaging",
    "ninja",
    "timm==0.4.12",
    "einops==0.7.0",
    "transformers==4.36.1",
    "opencv-python-headless",
    "matplotlib",
    "pandas",
    "numpy",
    "torchvision",
    "tqdm",
    "scikit-learn",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

VIDEOMAMBA_REPO_URL = "https://github.com/OpenGVLab/VideoMamba.git"
VIDEOMAMBA_COMMIT = "37355c26d0ae99ca2459f6d4044a5f509031a79f"

VIDEOMAMBA_REPO = Path("/kaggle/working/VideoMamba")
MAMBA_ROOT = VIDEOMAMBA_REPO / "mamba"
MAMBA_PACKAGE_DIR = MAMBA_ROOT / "mamba_ssm"
VIDEOMAMBA_PATH = VIDEOMAMBA_REPO / "videomamba" / "video_sm"

if VIDEOMAMBA_REPO.exists():
    shutil.rmtree(VIDEOMAMBA_REPO)

print("Cloning pinned VideoMamba source ...")
subprocess.run(
    ["git", "clone", VIDEOMAMBA_REPO_URL, str(VIDEOMAMBA_REPO)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(VIDEOMAMBA_REPO), "fetch", "origin", VIDEOMAMBA_COMMIT],
    check=True,
)
subprocess.run(
    ["git", "-C", str(VIDEOMAMBA_REPO), "checkout", "--force", VIDEOMAMBA_COMMIT],
    check=True,
)

current_commit = subprocess.check_output(
    ["git", "-C", str(VIDEOMAMBA_REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

assert current_commit == VIDEOMAMBA_COMMIT, current_commit
assert MAMBA_PACKAGE_DIR.is_dir(), MAMBA_PACKAGE_DIR
assert VIDEOMAMBA_PATH.is_dir(), VIDEOMAMBA_PATH

for p in [str(VIDEOMAMBA_PATH), str(MAMBA_ROOT)]:
    while p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

print("Python          :", sys.version.split()[0])
print("VideoMamba     :", current_commit)
print("MAMBA_ROOT     :", MAMBA_ROOT)
print("VIDEOMAMBA_PATH:", VIDEOMAMBA_PATH)

### 0.1 Robust PyTorch-reference Mamba fallback

This cell is adapted directly from the supplied benchmark notebook so the model can still be constructed when native binary wheels are unavailable.

In [ ]:
import types
import random
import importlib
import importlib.machinery

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange, repeat
from torchvision import transforms
from IPython.display import display
from tqdm.auto import tqdm

# -------------------------------------------------------------------------
# Clean previous Mamba imports so this cell can be rerun.
# -------------------------------------------------------------------------
for name in list(sys.modules):
    if name == "mamba_ssm" or name.startswith("mamba_ssm."):
        del sys.modules[name]

for name in ["causal_conv1d", "causal_conv1d_cuda", "selective_scan_cuda"]:
    sys.modules.pop(name, None)

importlib.invalidate_caches()

# -------------------------------------------------------------------------
# Source-package namespace.
# -------------------------------------------------------------------------
mamba_pkg = types.ModuleType("mamba_ssm")
mamba_pkg.__file__ = str(MAMBA_PACKAGE_DIR / "__init__.py")
mamba_pkg.__path__ = [str(MAMBA_PACKAGE_DIR)]
mamba_pkg.__package__ = "mamba_ssm"

mamba_spec = importlib.machinery.ModuleSpec(
    name="mamba_ssm",
    loader=None,
    is_package=True,
)
mamba_spec.submodule_search_locations = [str(MAMBA_PACKAGE_DIR)]
mamba_pkg.__spec__ = mamba_spec
sys.modules["mamba_ssm"] = mamba_pkg

# -------------------------------------------------------------------------
# PyTorch fallback for causal_conv1d.
# -------------------------------------------------------------------------
causal_mod = types.ModuleType("causal_conv1d")

def causal_conv1d_fn(x, weight, bias=None, activation=None):
    seqlen = x.shape[-1]
    dim = x.shape[1]
    width = weight.shape[-1]

    if weight.ndim == 2:
        w = weight.unsqueeze(1)
    elif weight.ndim == 3:
        w = weight
    else:
        raise ValueError(f"Invalid causal_conv1d weight shape: {tuple(weight.shape)}")

    out = F.conv1d(
        x,
        w,
        bias=bias,
        padding=width - 1,
        groups=dim,
    )[..., :seqlen]

    if activation in ["silu", "swish"]:
        out = F.silu(out)
    return out

def causal_conv1d_update(*args, **kwargs):
    raise NotImplementedError(
        "causal_conv1d_update is not used in batch VideoMamba inference."
    )

causal_mod.causal_conv1d_fn = causal_conv1d_fn
causal_mod.causal_conv1d_update = causal_conv1d_update
sys.modules["causal_conv1d"] = causal_mod
sys.modules["causal_conv1d_cuda"] = types.ModuleType("causal_conv1d_cuda")
sys.modules["selective_scan_cuda"] = types.ModuleType("selective_scan_cuda")

# -------------------------------------------------------------------------
# PyTorch fallback for RMSNorm/Triton layernorm.
# -------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5, device=None, dtype=None):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model, device=device, dtype=dtype))
        self.register_parameter("bias", None)

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rms_norm_fn(
    x,
    weight,
    bias=None,
    residual=None,
    eps=1e-5,
    prenorm=False,
    residual_in_fp32=False,
):
    if residual is not None:
        if residual_in_fp32:
            residual = residual.float()
        x = x + residual
        res = x
    else:
        res = x

    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps) * weight
    if bias is not None:
        normed = normed + bias

    if prenorm:
        return normed, res
    return normed

triton_ln = types.ModuleType("mamba_ssm.ops.triton.layernorm")
triton_ln.RMSNorm = RMSNorm
triton_ln.layer_norm_fn = None
triton_ln.rms_norm_fn = rms_norm_fn
sys.modules["mamba_ssm.ops.triton.layernorm"] = triton_ln

# Compatibility shim for some transformers versions.
try:
    import transformers.generation
    if not hasattr(transformers.generation, "GreedySearchDecoderOnlyOutput"):
        transformers.generation.GreedySearchDecoderOnlyOutput = object
        transformers.generation.SampleDecoderOnlyOutput = object
except Exception:
    pass

import mamba_ssm.ops.selective_scan_interface as ssi
import mamba_ssm.modules.mamba_simple as ms

def mamba_inner_fn_no_out_proj(
    xz,
    conv1d_weight,
    conv1d_bias,
    x_proj_weight,
    delta_proj_weight,
    A,
    B=None,
    C=None,
    D=None,
    delta_bias=None,
    B_proj_bias=None,
    C_proj_bias=None,
    delta_softplus=True,
):
    L = xz.shape[-1]
    delta_rank = delta_proj_weight.shape[1]
    d_state = A.shape[-1] * (1 if not A.is_complex() else 2)

    x, z = xz.chunk(2, dim=1)

    x = causal_conv1d_fn(
        x,
        rearrange(conv1d_weight, "d 1 w -> d w"),
        conv1d_bias,
        "silu",
    )

    x_dbl = F.linear(
        rearrange(x, "b d l -> (b l) d"),
        x_proj_weight,
    )

    delta = delta_proj_weight @ x_dbl[:, :delta_rank].t()
    delta = rearrange(delta, "d (b l) -> b d l", l=L)

    if B is None:
        B = x_dbl[:, delta_rank:delta_rank + d_state]
        if B_proj_bias is not None:
            B = B + B_proj_bias.to(dtype=B.dtype)
        if not A.is_complex():
            B = rearrange(B, "(b l) dstate -> b dstate l", l=L).contiguous()
        else:
            B = rearrange(
                B, "(b l) (dstate two) -> b dstate (l two)",
                l=L, two=2
            ).contiguous()

    if C is None:
        C = x_dbl[:, -d_state:]
        if C_proj_bias is not None:
            C = C + C_proj_bias.to(dtype=C.dtype)
        if not A.is_complex():
            C = rearrange(C, "(b l) dstate -> b dstate l", l=L).contiguous()
        else:
            C = rearrange(
                C, "(b l) (dstate two) -> b dstate (l two)",
                l=L, two=2
            ).contiguous()

    return ssi.selective_scan_ref(
        x,
        delta,
        A,
        B,
        C,
        D,
        z=z,
        delta_bias=delta_bias,
        delta_softplus=delta_softplus,
    )

# Reference implementation for robust model loading.
ssi.selective_scan_fn = ssi.selective_scan_ref
ssi.mamba_inner_fn = ssi.mamba_inner_ref
ssi.bimamba_inner_fn = ssi.bimamba_inner_ref
ssi.mamba_inner_fn_no_out_proj = mamba_inner_fn_no_out_proj

ms.selective_scan_fn = ssi.selective_scan_ref
ms.mamba_inner_fn = ssi.mamba_inner_ref
ms.bimamba_inner_fn = ssi.bimamba_inner_ref
ms.mamba_inner_fn_no_out_proj = mamba_inner_fn_no_out_proj

from models import videomamba_tiny

MAMBA_BACKEND = "pytorch_reference"

print("OpenCV         :", cv2.__version__)
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
print("VideoMamba     : imported")
print("Mamba backend  :", MAMBA_BACKEND)


## 1. Global configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# ---------------------------------------------------------------------
# CTPGesture_v1
# ---------------------------------------------------------------------
INPUT_ROOT = Path("/kaggle/input")

TEST_IDS = [
    "002", "004", "008", "010", "012", "014",
    "016", "018", "101", "102", "104",
]

CLASS_DESCRIPTIONS = [
    "No Gesture / Stand in attention",
    "Stop",
    "Forward",
    "Left Turn",
    "Left Turn Waiting",
    "Right Turn",
    "Lane Changing",
    "Slow Down",
    "Pull Over",
]

CLASS_CODES = [str(i) for i in range(9)]
CLASS_NAMES = [
    f"{i} - {name}"
    for i, name in enumerate(CLASS_DESCRIPTIONS)
]

NUM_CLASSES = 9

# ---------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------
NUM_FRAMES = 16
IMG_SIZE = 224
LOAD_CHECKPOINT = True

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        torch.set_float32_matmul_precision("highest")
    except Exception:
        pass

# ---------------------------------------------------------------------
# Continuous recognition
# ---------------------------------------------------------------------
WINDOW_SIZE = 32
WINDOW_STEP = 8
MODEL_FRAMES = 16

assert WINDOW_SIZE % MODEL_FRAMES == 0
assert 0 < WINDOW_STEP <= WINDOW_SIZE

OVERLAP_RATIO = 1.0 - WINDOW_STEP / WINDOW_SIZE

# Candidate only. The ablation section determines the best CTPGesture
# post-processing configuration independently.
ENABLE_SMOOTHING = False
T_AVG_SEC = 0.0
MIN_GESTURE_SEC = 0.0
FILL_SAME_NEIGHBORS = False

# Evaluation tolerance for temporal segment matching.
THETA_SEC = 2.0

# ---------------------------------------------------------------------
# Evaluation controls
# ---------------------------------------------------------------------
ISOLATED_MAX_SAMPLES = None
CONTINUOUS_MAX_VIDEOS = None
DEBUG_VIDEO_ID = "002"

# ---------------------------------------------------------------------
# Performance controls
# ---------------------------------------------------------------------
# FP32 inference stays unchanged. These values affect execution speed only.
ISOLATED_BATCH_SIZE = 4
CONTINUOUS_BATCH_SIZE = 4

# Full Save-Version runs should normally skip the separate debug pass.
# If enabled, Section 7.2 will reuse the debug cache instead of inferring
# DEBUG_VIDEO_ID a second time.
RUN_DEBUG_VIDEO = False
REUSE_DEBUG_CACHE = True

# Save one raw cache file per continuous video. Re-running Section 7.2 in
# the same Kaggle session can then reuse completed videos.
REUSE_PER_VIDEO_CACHE = False  # P1 full rerun: prevent stale predictions from other checkpoints
SAVE_PER_VIDEO_CACHE = True
PER_VIDEO_CACHE_DIR = None

RUN_ISOLATED = True
RUN_CONTINUOUS = True
RUN_ABLATION = False
RUN_COMPLEXITY = False

OUTPUT_DIR = Path(
    "/kaggle/working/ctpgesture_videomamba_p1_evaluation"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PER_VIDEO_CACHE_DIR = OUTPUT_DIR / "per_video_raw_cache"
PER_VIDEO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE            :", DEVICE)
print("Classes           :", NUM_CLASSES)
print("Model frames      :", NUM_FRAMES)
print("Window size       :", WINDOW_SIZE)
print("Window step       :", WINDOW_STEP)
print("Window overlap    :", f"{OVERLAP_RATIO*100:.2f}%")
print("Window duration @15 FPS:", f"{WINDOW_SIZE/15:.3f} s")
print("Output            :", OUTPUT_DIR)

print("Isolated batch    :", ISOLATED_BATCH_SIZE)
print("Continuous batch  :", CONTINUOUS_BATCH_SIZE)
print("Debug pass        :", RUN_DEBUG_VIDEO)
print("Per-video cache   :", PER_VIDEO_CACHE_DIR)


## 2. Locate dataset, checkpoint, and optional native CUDA wheels

In [ ]:
# ---------------------------------------------------------------------
# Locate CTPGesture_v1
# ---------------------------------------------------------------------
dataset_candidates = [
    p for p in INPUT_ROOT.glob("**/CTPGesture_v1/police_gesture_v1")
    if (p / "train").is_dir() and (p / "test").is_dir()
]

if not dataset_candidates:
    dataset_candidates = [
        p for p in INPUT_ROOT.glob("**/police_gesture_v1")
        if (p / "train").is_dir() and (p / "test").is_dir()
    ]

if not dataset_candidates:
    raise FileNotFoundError(
        "Cannot find CTPGesture_v1/police_gesture_v1 under /kaggle/input."
    )

dataset_candidates = sorted(
    set(dataset_candidates),
    key=lambda p: (len(p.parts), str(p)),
)

DATA_ROOT = dataset_candidates[0]
TEST_DIR = DATA_ROOT / "test"

# ---------------------------------------------------------------------
# Locate checkpoint-best.pth
# ---------------------------------------------------------------------
preferred_checkpoint = Path(
    "/kaggle/input/models/hoanganh94/"
    "videomambactpgesturev1best/"
    "pytorch/tiny-16x2-224-fp32/1/"
    "checkpoint-best.pth"
)

if preferred_checkpoint.is_file():
    CHECKPOINT_PATH = preferred_checkpoint
else:
    checkpoint_candidates = sorted(
        INPUT_ROOT.rglob("checkpoint-best.pth"),
        key=lambda p: (
            "videomambactpgesturev1best" not in str(p).lower(),
            "tiny-16x2-224-fp32" not in str(p).lower(),
            "/models/" not in str(p).lower(),
            len(p.parts),
            str(p),
        ),
    )

    if not checkpoint_candidates:
        raise FileNotFoundError(
            "Cannot find checkpoint-best.pth under /kaggle/input."
        )

    CHECKPOINT_PATH = checkpoint_candidates[0]

# P1: native wheel pair is discovered by the next manifest validation cell.
CAUSAL_WHEEL = None
MAMBA_WHEEL = None
ACTIVATE_NATIVE_MAMBA_CUDA = False

print("DATA_ROOT        :", DATA_ROOT)
print("TEST_DIR         :", TEST_DIR)
print("CHECKPOINT_PATH  :", CHECKPOINT_PATH)
print("Checkpoint size  :", f"{CHECKPOINT_PATH.stat().st_size/1024**2:.2f} MiB")
print("Python           :", sys.version.split()[0])
print("Native CUDA: resolved after manifest verification")

if CAUSAL_WHEEL:
    print("causal_conv1d    :", CAUSAL_WHEEL)
if MAMBA_WHEEL:
    print("mamba_ssm        :", MAMBA_WHEEL)

# Manifest resolver in the next cell validates the attached wheels.


### P1 Wheel selection: validated runtime-specific native Mamba

Search attached Kaggle Input datasets for builder `wheel_manifest.json`, validate wheels and compatibility, then activate native CUDA. CPython 3.13 cp313 wheels must match Torch/CUDA/ABI/SM exactly.


In [ ]:
# P1 startup gate: select wheel pair using the builder's validated manifest.
# Attach the NEW Kaggle wheel dataset as an Input before running this notebook.
# This deliberately rejects cp312 binaries in a cp313 Python environment.
import sys, json, hashlib, re
from pathlib import Path
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
import torch

P1_EXPECTED_VIDEOMAMBA_COMMIT = "37355c26d0ae99ca2459f6d4044a5f509031a79f"
P1_WHEEL_DATASET_HINT = ""  # Optional substring of the mounted Kaggle dataset path.
P1_REQUIRE_NATIVE_CUDA = True  # Fail early instead of silently using slow reference inference.

def _p1_sha256(file):
    h = hashlib.sha256()
    with Path(file).open("rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _p1_inspect_manifest(manifest_path):
    m = json.loads(manifest_path.read_text(encoding="utf-8"))
    e = m["environment"]
    python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    if e.get("python_tag") != python_tag:
        raise ValueError(f"Python ABI {e.get('python_tag')} != {python_tag}")
    if e.get("torch_version") != torch.__version__:
        raise ValueError(f"PyTorch {e.get('torch_version')} != {torch.__version__}")
    if str(e.get("torch_cuda")) != str(torch.version.cuda):
        raise ValueError(f"PyTorch CUDA {e.get('torch_cuda')} != {torch.version.cuda}")
    abi = int(bool(torch._C._GLIBCXX_USE_CXX11_ABI))
    if int(e.get("torch_cxx11_abi", -1)) != abi:
        raise ValueError("PyTorch C++11 ABI mismatch")
    expected_commit = P1_EXPECTED_VIDEOMAMBA_COMMIT
    if m.get("source", {}).get("commit") != expected_commit:
        raise ValueError("VideoMamba wheel source commit differs from pinned model source")
    if not m.get("verification", {}).get("cuda_smoke_test_passed", False):
        raise ValueError("Wheel builder CUDA smoke test did not pass")
    if not torch.cuda.is_available():
        raise ValueError("CUDA GPU is not enabled")
    actual_sms = {"".join(map(str,torch.cuda.get_device_capability(i)))
                  for i in range(torch.cuda.device_count())}
    if not actual_sms.issubset(set(map(str,e.get("target_sms",[])))):
        raise ValueError(f"GPU targets {actual_sms} not supported by {e.get('target_sms')}")
    compatible = set(sys_tags())
    selected = {}
    for package in ("causal_conv1d", "mamba_ssm"):
        item = m["wheels"][package]
        candidate = manifest_path.parent / item["file"]
        if not candidate.is_file():
            raise ValueError(f"Wheel file missing: {candidate}")
        if _p1_sha256(candidate) != item["sha256"]:
            raise ValueError(f"Wheel checksum mismatch: {candidate}")
        name, _, _, tags = parse_wheel_filename(candidate.name)
        if name.replace("-", "_") != package:
            raise ValueError(f"Wrong wheel package: {candidate.name}")
        if not (compatible & tags):
            raise ValueError(f"Incompatible Python/platform wheel tags: {candidate.name}")
        selected[package] = candidate
    return selected, m

assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle first."
P1_WHEEL_CANDIDATES = sorted(Path("/kaggle/input").rglob("wheel_manifest.json"))
if P1_WHEEL_DATASET_HINT:
    P1_WHEEL_CANDIDATES = [p for p in P1_WHEEL_CANDIDATES
                           if P1_WHEEL_DATASET_HINT.lower() in str(p).lower()]
P1_MATCHES, P1_REJECTIONS = [], []
for manifest_path in P1_WHEEL_CANDIDATES:
    try:
        pair, manifest_data = _p1_inspect_manifest(manifest_path)
        P1_MATCHES.append((manifest_path, pair, manifest_data))
    except Exception as exc:
        P1_REJECTIONS.append((str(manifest_path), str(exc)))

print("Python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| torch CUDA:", torch.version.cuda)
print("Found wheel manifests:", len(P1_WHEEL_CANDIDATES),
      "| matches:", len(P1_MATCHES))
for path, reason in P1_REJECTIONS[:8]:
    print("[REJECTED]", path, "because", reason)
if len(P1_MATCHES) != 1:
    raise RuntimeError(
        f"Expected exactly one ABI/torch/CUDA/SM-compatible verified wheel manifest; "
        f"found {len(P1_MATCHES)}. Attach the NEW wheel Dataset and optionally set "
        f"P1_WHEEL_DATASET_HINT to its path. Never rename/install cp312 as cp313.")
P1_WHEEL_MANIFEST, P1_WHEEL_PAIR, P1_WHEEL_METADATA = P1_MATCHES[0]
CAUSAL_WHEEL = P1_WHEEL_PAIR["causal_conv1d"]
MAMBA_WHEEL = P1_WHEEL_PAIR["mamba_ssm"]
ACTIVATE_NATIVE_MAMBA_CUDA = True
print("Selected wheel manifest:", P1_WHEEL_MANIFEST)
print("Causal wheel:", CAUSAL_WHEEL.name)
print("Mamba wheel :", MAMBA_WHEEL.name)


## 3. Parse CTPGesture annotations and verify the official test split

In [ ]:
import re
import cv2
import numpy as np
import pandas as pd

def read_ctp_labels(csv_path):
    text = Path(csv_path).read_text(
        encoding="utf-8"
    ).strip()

    tokens = [
        token
        for token in re.split(r"[\s,]+", text)
        if token
    ]

    labels = np.asarray(
        [int(token) for token in tokens],
        dtype=np.int64,
    )

    if len(labels) == 0:
        raise RuntimeError(
            f"Empty annotation: {csv_path}"
        )

    if labels.min() < 0 or labels.max() >= NUM_CLASSES:
        raise ValueError(
            f"Invalid labels in {csv_path}: "
            f"{labels.min()}..{labels.max()}"
        )

    return labels

def labels_to_contiguous_segments(labels):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    if len(labels) == 0:
        return []

    starts = np.r_[
        0,
        np.flatnonzero(
            labels[1:] != labels[:-1]
        ) + 1,
    ]

    ends = np.r_[
        starts[1:] - 1,
        len(labels) - 1,
    ]

    return [
        {
            "label": int(labels[start]),
            "start": int(start),
            "end": int(end),
        }
        for start, end in zip(starts, ends)
    ]

audit_rows = []
segment_rows = []

for video_id in TEST_IDS:
    video_path = TEST_DIR / f"{video_id}.mp4"
    csv_path = TEST_DIR / f"{video_id}.csv"

    if not video_path.is_file():
        raise FileNotFoundError(video_path)

    if not csv_path.is_file():
        raise FileNotFoundError(csv_path)

    labels = read_ctp_labels(csv_path)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {video_path}")

    video_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )
    fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )
    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )
    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )
    cap.release()

    diff = video_frames - len(labels)

    if abs(diff) > 1:
        raise RuntimeError(
            f"{video_id}: video frames={video_frames}, "
            f"CSV labels={len(labels)}, diff={diff}"
        )

    audit_rows.append({
        "video_id": video_id,
        "video_path": str(video_path),
        "csv_path": str(csv_path),
        "video_frames": video_frames,
        "csv_labels": len(labels),
        "frame_diff": diff,
        "fps": fps,
        "width": width,
        "height": height,
    })

    for segment_id, seg in enumerate(
        labels_to_contiguous_segments(labels)
    ):
        segment_rows.append({
            "sample_id": f"{video_id}_seg{segment_id:04d}",
            "video_id": video_id,
            "video_path": str(video_path),
            "csv_path": str(csv_path),
            "label": int(seg["label"]),
            "t_start": int(seg["start"]),
            "t_end": int(seg["end"]),
            "duration_frames": int(
                seg["end"] - seg["start"] + 1
            ),
        })

audit_df = pd.DataFrame(audit_rows)
test_segments = pd.DataFrame(segment_rows)

print("=" * 72)
print("OFFICIAL CTPGesture TEST SPLIT")
print("=" * 72)
print("Videos            :", len(audit_df))
print("Frames            :", int(audit_df["csv_labels"].sum()))
print("Temporal segments :", len(test_segments))
print("Class-0 segments  :", int((test_segments["label"] == 0).sum()))
print("Gesture segments  :", int((test_segments["label"] != 0).sum()))

display(audit_df)

segment_distribution = (
    test_segments
    .groupby("label")
    .size()
    .rename("segments")
    .to_frame()
)
segment_distribution["class_name"] = [
    CLASS_DESCRIPTIONS[i]
    for i in segment_distribution.index
]
display(segment_distribution)

if len(audit_df) != 11:
    print("[WARNING] Expected 11 official test videos.")

if len(test_segments) != 1728:
    print(
        "[WARNING] Previously verified split had 1728 segments; "
        f"current parse produced {len(test_segments)}."
    )

test_segments.to_csv(
    OUTPUT_DIR / "ctpgesture_official_test_segments.csv",
    index=False,
)
audit_df.to_csv(
    OUTPUT_DIR / "ctpgesture_official_test_audit.csv",
    index=False,
)

## 4. Load the fine-tuned VideoMamba-Tiny checkpoint

In [ ]:
import importlib
import subprocess
import torch

def load_videomamba_model(
    checkpoint_path=CHECKPOINT_PATH,
    num_classes=NUM_CLASSES,
    device=DEVICE,
):
    model_vm = videomamba_tiny(
        img_size=IMG_SIZE,
        num_frames=NUM_FRAMES,
        kernel_size=1,
        num_classes=num_classes,
        drop_path_rate=0.1,
        fc_drop_rate=0.3,
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    state_dict = checkpoint.get(
        "model",
        checkpoint,
    )

    clean_state_dict = {}
    for key, value in state_dict.items():
        name = (
            key[7:]
            if key.startswith("module.")
            else key
        )
        clean_state_dict[name] = value

    model_vm.load_state_dict(
        clean_state_dict,
        strict=True,
    )

    model_vm = (
        model_vm
        .float()
        .to(device)
        .eval()
    )

    return model_vm, checkpoint

# Build with the robust reference backend first.
model, checkpoint = load_videomamba_model()
MAMBA_BACKEND = "pytorch_reference"

# Optional native CUDA activation on CPython 3.12.
if ACTIVATE_NATIVE_MAMBA_CUDA:
    print("Installing native Mamba CUDA wheels ...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "--no-deps",
            str(CAUSAL_WHEEL),
            str(MAMBA_WHEEL),
        ],
        check=True,
    )

    for name in list(sys.modules):
        if (
            name == "mamba_ssm"
            or name.startswith("mamba_ssm.")
            or name == "causal_conv1d"
            or name.startswith("causal_conv1d.")
            or name == "causal_conv1d_cuda"
            or name == "selective_scan_cuda"
        ):
            sys.modules.pop(name, None)

    importlib.invalidate_caches()

    try:
        import causal_conv1d_cuda
        import selective_scan_cuda
        import mamba_ssm
        import mamba_ssm.ops.selective_scan_interface as ssi_native
        import mamba_ssm.modules.mamba_simple as ms_native

        ssi_native = importlib.reload(
            ssi_native
        )
        ms_native = importlib.reload(
            ms_native
        )

        if (
            ssi_native.selective_scan_fn
            is ssi_native.selective_scan_ref
        ):
            raise RuntimeError(
                "Native selective_scan_fn still points "
                "to the reference path."
            )

        import models.videomamba as vm_mod

        vm_mod.Mamba = ms_native.Mamba
        videomamba_tiny = (
            vm_mod.videomamba_tiny
        )

        model, checkpoint = (
            load_videomamba_model()
        )
        MAMBA_BACKEND = "native_cuda"

    except Exception as exc:
        print(
            "[WARNING] Native CUDA activation failed:",
            repr(exc),
        )
        print(
            "Continuing with the reference model."
        )

checkpoint_epoch = checkpoint.get(
    "epoch",
    None,
)

print("=" * 72)
print("MODEL READY")
print("=" * 72)
print("Checkpoint       :", CHECKPOINT_PATH)
print("Checkpoint epoch :", checkpoint_epoch)
print("Backend          :", MAMBA_BACKEND)
print("Device           :", next(model.parameters()).device)
print("Dtype            :", next(model.parameters()).dtype)
print("Parameters       :", f"{sum(p.numel() for p in model.parameters()):,}")

### P1 Native CUDA + checkpoint smoke test

Fail fast on wheel import/native mismatch, run one dummy CUDA model forward, save model/checkpoint environment audit.


In [ ]:
# P1 fail-fast CUDA and checkpoint verification before the costly full test split.
import torch, hashlib, json
from pathlib import Path

assert torch.cuda.is_available(), "GPU not enabled."
assert MAMBA_BACKEND == "native_cuda", (
    "Native Mamba backend did not activate. Check wheel manifest/runtime compatibility.")
P1_MODEL_CHECKPOINT = Path(CHECKPOINT_PATH)
assert P1_MODEL_CHECKPOINT.is_file(), "Fine-tuned checkpoint missing."
assert next(model.parameters()).device.type == "cuda", "Model is not using CUDA."
with torch.inference_mode():
    p1_test_logits = model(torch.zeros(
        (1, 3, NUM_FRAMES, IMG_SIZE, IMG_SIZE), device=DEVICE, dtype=torch.float32))
torch.cuda.synchronize()
assert p1_test_logits.shape == (1, NUM_CLASSES), (
    f"Unexpected inference logits shape: {p1_test_logits.shape}")
assert torch.isfinite(p1_test_logits).all(), "NaN/Inf in CUDA output."
P1_CHECKPOINT_SHA256 = _p1_sha256(P1_MODEL_CHECKPOINT)
P1_RUNTIME_AUDIT = {
    "python": sys.version.split()[0], "torch": torch.__version__,
    "torch_cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "native_backend": MAMBA_BACKEND,
    "wheel_manifest": str(P1_WHEEL_MANIFEST),
    "checkpoint": str(P1_MODEL_CHECKPOINT),
    "checkpoint_sha256": P1_CHECKPOINT_SHA256,
    "videomamba_source_commit": P1_EXPECTED_VIDEOMAMBA_COMMIT,
    "frames": NUM_FRAMES, "image_size": IMG_SIZE, "num_classes": NUM_CLASSES
}
P1_AUDIT_FILE = Path("/kaggle/working/p1_cuda_runtime_audit.json")
P1_AUDIT_FILE.write_text(json.dumps(P1_RUNTIME_AUDIT, indent=2), encoding="utf-8")
print("[PASS] Native CUDA model forward smoke test; runtime audit:", P1_AUDIT_FILE)


## 5. Common preprocessing and inference helpers

In [ ]:
import cv2
import time
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

def sample_indices(total_frames, num_frames=NUM_FRAMES):
    if total_frames <= 0:
        return np.zeros(num_frames, dtype=np.int64)

    return np.linspace(
        0,
        total_frames - 1,
        num_frames,
    ).round().astype(np.int64)

def preprocess_rgb_frame(frame_rgb):
    """
    RGB uint8 HWC -> normalized float32 CHW on CPU.
    """
    return transform(frame_rgb)

def frames_to_clip_tensor(frames_np):
    """
    THWC uint8 -> CTHW float32 on CPU.
    Kept for one-off inference/debug compatibility.
    """
    tensor = torch.stack([
        preprocess_rgb_frame(frame)
        for frame in frames_np
    ])
    return tensor.permute(1, 0, 2, 3).contiguous()

def clips_to_probabilities(
    model,
    clips_cthw,
    device=DEVICE,
):
    """
    Batched FP32 inference.

    Parameters
    ----------
    clips_cthw : list[Tensor]
        Each tensor has shape C,T,H,W and is stored on CPU.

    Returns
    -------
    probs : np.ndarray
        Shape B,num_classes.
    model_forward_sec : float
        GPU-synchronized model forward time when CUDA is used.
    """
    if len(clips_cthw) == 0:
        return (
            np.empty((0, NUM_CLASSES), dtype=np.float32),
            0.0,
        )

    batch = torch.stack(
        clips_cthw,
        dim=0,
    ).to(
        device=device,
        dtype=torch.float32,
        non_blocking=True,
    )

    try:
        if device.type == "cuda":
            torch.cuda.synchronize(device)

        t0 = time.perf_counter()

        with torch.inference_mode():
            logits = model(batch)
            probs = F.softmax(logits, dim=1)

        if device.type == "cuda":
            torch.cuda.synchronize(device)

        elapsed = time.perf_counter() - t0

        probs_np = (
            probs.detach()
            .float()
            .cpu()
            .numpy()
            .astype(np.float32, copy=False)
        )

        del batch, logits, probs
        return probs_np, float(elapsed)

    except RuntimeError as exc:
        # Automatic fallback if a chosen batch size is too large.
        if (
            "out of memory" in str(exc).lower()
            and len(clips_cthw) > 1
        ):
            if device.type == "cuda":
                torch.cuda.empty_cache()

            mid = len(clips_cthw) // 2

            left_probs, left_sec = (
                clips_to_probabilities(
                    model,
                    clips_cthw[:mid],
                    device,
                )
            )

            right_probs, right_sec = (
                clips_to_probabilities(
                    model,
                    clips_cthw[mid:],
                    device,
                )
            )

            return (
                np.concatenate(
                    [left_probs, right_probs],
                    axis=0,
                ),
                left_sec + right_sec,
            )

        raise

def run_inference(model, frames_np, device=DEVICE):
    """
    Compatibility helper for a single clip.
    """
    clip = frames_to_clip_tensor(frames_np)
    probs, _ = clips_to_probabilities(
        model,
        [clip],
        device,
    )
    probs = probs[0]

    pred_idx = int(np.argmax(probs))
    confidence = float(probs[pred_idx])

    return pred_idx, confidence, probs

print("[OK] Optimized preprocessing / batched inference helpers ready.")

## 6. Isolated gesture classification

Each contiguous annotated interval is treated as one isolated clip.

Exactly 16 frames are uniformly sampled across the interval, matching the
methodology of the supplied benchmark notebook.

The model is run once per annotated segment.

The cell computes:

1. all-class report (`0..8`);
2. gesture-only report (`1..8`);
3. per-class precision/recall/F1;
4. confusion matrix;
5. prediction CSV.

The gesture-only report does **not** force predictions into classes `1..8`.
A true gesture predicted as class `0` remains an error.

In [ ]:
import time
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

def classification_metric_dict(
    y_true,
    y_pred,
    labels_for_macro,
):
    return {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_precision": float(
            precision_score(
                y_true,
                y_pred,
                labels=labels_for_macro,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true,
                y_pred,
                labels=labels_for_macro,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=labels_for_macro,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=labels_for_macro,
                average="weighted",
                zero_division=0,
            )
        ),
    }

def run_isolated_video_sequential(
    model,
    video_df,
    batch_size=ISOLATED_BATCH_SIZE,
):
    """
    Decode each video sequentially ONCE.

    Segments in CTPGesture are contiguous and time ordered, so there is no need
    to reopen the MP4 or seek randomly for every isolated sample.
    """
    video_df = (
        video_df
        .sort_values(["t_start", "t_end"])
        .reset_index(drop=True)
    )

    video_path = str(
        video_df.iloc[0]["video_path"]
    )

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(
            f"Cannot open video: {video_path}"
        )

    current_frame_idx = 0
    pending_clips = []
    pending_meta = []
    output_records = []

    decode_preprocess_sec = 0.0
    model_forward_sec = 0.0

    def flush_pending():
        nonlocal pending_clips
        nonlocal pending_meta
        nonlocal model_forward_sec
        nonlocal output_records

        if not pending_clips:
            return

        probs_batch, forward_sec = (
            clips_to_probabilities(
                model,
                pending_clips,
                DEVICE,
            )
        )

        model_forward_sec += forward_sec

        for meta, probs in zip(
            pending_meta,
            probs_batch,
        ):
            pred_idx = int(
                np.argmax(probs)
            )
            conf = float(
                probs[pred_idx]
            )

            record = {
                **meta,
                "pred_idx": pred_idx,
                "pred_name": (
                    CLASS_DESCRIPTIONS[
                        pred_idx
                    ]
                ),
                "confidence": conf,
                "correct": (
                    pred_idx
                    == meta["gt_idx"]
                ),
            }

            for class_idx, prob in enumerate(
                probs
            ):
                record[
                    f"prob_{class_idx}"
                ] = float(prob)

            output_records.append(record)

        pending_clips = []
        pending_meta = []

    for row in video_df.itertuples(
        index=False
    ):
        start_f = int(row.t_start)
        end_f = int(row.t_end)

        clip_len = (
            end_f - start_f + 1
        )

        rel_idx = sample_indices(
            clip_len,
            NUM_FRAMES,
        )

        target_idx = (
            start_f + rel_idx
        ).astype(np.int64)

        # Because segments are processed chronologically, decode forward only.
        target_unique = set(
            target_idx.tolist()
        )
        frame_tensor_by_idx = {}

        t_decode = time.perf_counter()

        while (
            current_frame_idx
            <= int(target_idx.max())
        ):
            ok, frame_bgr = cap.read()

            if not ok:
                break

            if current_frame_idx in target_unique:
                frame_rgb = cv2.cvtColor(
                    frame_bgr,
                    cv2.COLOR_BGR2RGB,
                )

                frame_tensor_by_idx[
                    current_frame_idx
                ] = preprocess_rgb_frame(
                    frame_rgb
                )

            current_frame_idx += 1

        decode_preprocess_sec += (
            time.perf_counter()
            - t_decode
        )

        if not frame_tensor_by_idx:
            raise RuntimeError(
                f"No sampled frames decoded "
                f"for {row.sample_id}"
            )

        # Duplicate rounded indices are allowed.
        frames = []
        last_tensor = None

        for idx in target_idx.tolist():
            if idx in frame_tensor_by_idx:
                tensor = frame_tensor_by_idx[
                    idx
                ]
                last_tensor = tensor
            elif last_tensor is not None:
                tensor = last_tensor
            else:
                # This should not normally happen.
                tensor = next(
                    iter(
                        frame_tensor_by_idx
                        .values()
                    )
                )

            frames.append(tensor)

        clip = (
            torch.stack(frames, dim=0)
            .permute(1, 0, 2, 3)
            .contiguous()
        )

        pending_clips.append(clip)
        pending_meta.append({
            "sample_id": row.sample_id,
            "video_id": row.video_id,
            "t_start": start_f,
            "t_end": end_f,
            "duration_frames": int(
                row.duration_frames
            ),
            "gt_idx": int(row.label),
            "gt_name": (
                CLASS_DESCRIPTIONS[
                    int(row.label)
                ]
            ),
        })

        if (
            len(pending_clips)
            >= int(batch_size)
        ):
            flush_pending()

    flush_pending()
    cap.release()

    return (
        output_records,
        decode_preprocess_sec,
        model_forward_sec,
    )

isolated_results = None
isolated_all_metrics = None
isolated_gesture_metrics = None

if RUN_ISOLATED:
    isolated_df = test_segments.copy()

    if ISOLATED_MAX_SAMPLES is not None:
        isolated_df = isolated_df.iloc[
            :int(ISOLATED_MAX_SAMPLES)
        ].copy()

    all_records = []
    total_decode_sec = 0.0
    total_forward_sec = 0.0

    wall_t0 = time.perf_counter()

    for video_id, video_df in tqdm(
        list(
            isolated_df.groupby(
                "video_id",
                sort=False,
            )
        ),
        desc="Isolated videos",
    ):
        (
            records,
            decode_sec,
            forward_sec,
        ) = run_isolated_video_sequential(
            model,
            video_df,
            ISOLATED_BATCH_SIZE,
        )

        all_records.extend(records)
        total_decode_sec += decode_sec
        total_forward_sec += forward_sec

    isolated_wall_sec = (
        time.perf_counter()
        - wall_t0
    )

    isolated_results = pd.DataFrame(
        all_records
    )

    # Restore deterministic original segment order.
    isolated_results = (
        isolated_results
        .merge(
            isolated_df[
                ["sample_id"]
            ].reset_index().rename(
                columns={
                    "index": "_order"
                }
            ),
            on="sample_id",
            how="left",
        )
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

    y_true_all = (
        isolated_results["gt_idx"]
        .to_numpy(dtype=np.int64)
    )
    y_pred_all = (
        isolated_results["pred_idx"]
        .to_numpy(dtype=np.int64)
    )

    isolated_all_metrics = (
        classification_metric_dict(
            y_true_all,
            y_pred_all,
            list(range(NUM_CLASSES)),
        )
    )

    gesture_mask = (
        isolated_results["gt_idx"] != 0
    )

    y_true_gesture = (
        isolated_results.loc[
            gesture_mask,
            "gt_idx",
        ].to_numpy(dtype=np.int64)
    )
    y_pred_gesture = (
        isolated_results.loc[
            gesture_mask,
            "pred_idx",
        ].to_numpy(dtype=np.int64)
    )

    isolated_gesture_metrics = (
        classification_metric_dict(
            y_true_gesture,
            y_pred_gesture,
            list(range(1, NUM_CLASSES)),
        )
    )

    print("=" * 72)
    print("ISOLATED — ALL CLASSES")
    print("=" * 72)
    print("Samples:", len(y_true_all))
    for key, value in isolated_all_metrics.items():
        print(
            f"{key:20s}: {value:.6f}"
        )

    print()
    print("=" * 72)
    print(
        "ISOLATED — GESTURE ONLY "
        "(GT classes 1..8)"
    )
    print("=" * 72)
    print("Samples:", len(y_true_gesture))
    for key, value in (
        isolated_gesture_metrics.items()
    ):
        print(
            f"{key:20s}: {value:.6f}"
        )

    print()
    print("Timing")
    print(
        f"  wall time          : "
        f"{isolated_wall_sec:.1f} s"
    )
    print(
        f"  decode/preprocess  : "
        f"{total_decode_sec:.1f} s"
    )
    print(
        f"  model forward      : "
        f"{total_forward_sec:.1f} s"
    )

    isolated_report = (
        classification_report(
            y_true_all,
            y_pred_all,
            labels=list(
                range(NUM_CLASSES)
            ),
            target_names=CLASS_NAMES,
            output_dict=True,
            digits=4,
            zero_division=0,
        )
    )

    isolated_report_df = pd.DataFrame(
        isolated_report
    ).T

    display(isolated_report_df)

    cm = confusion_matrix(
        y_true_all,
        y_pred_all,
        labels=list(
            range(NUM_CLASSES)
        ),
        normalize="true",
    )

    fig, ax = plt.subplots(
        figsize=(10, 8)
    )

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_CODES,
    ).plot(
        ax=ax,
        values_format=".2f",
        colorbar=True,
    )

    ax.set_title(
        "CTPGesture isolated — "
        "normalized confusion matrix"
    )
    plt.tight_layout()
    plt.show()

    isolated_results.to_csv(
        OUTPUT_DIR
        / "isolated_predictions.csv",
        index=False,
    )

    isolated_report_df.to_csv(
        OUTPUT_DIR
        / "isolated_classification_report.csv"
    )

else:
    print("[SKIP] RUN_ISOLATED=False")

### 6.1 P1: Accuracy@10/20/50/75/100% with GT-prefix inputs

Additional isolated prefix GPU inference (except 100% baseline reused). Both 9-class and GT-gesture-only metrics; **not** RSC causal Early@10.


In [ ]:
# P1 early isolated classification from GT segment prefixes.
# RSC Early@10 = correct command within ten causal output steps; NOT the same metric.
# Reuse original full-length test predictions at 100%, infer only 10/20/50/75%.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

P1_CTP_DIR = OUTPUT_DIR / "p1_metric_audit"
P1_CTP_DIR.mkdir(parents=True, exist_ok=True)
P1_PREFIX_FRACTIONS = (0.10, 0.20, 0.50, 0.75, 1.0)
RUN_P1_PREFIX = True
assert RUN_ISOLATED and isolated_results is not None
assert len(isolated_df) == len(isolated_results), "Mismatch in isolated test sample count."
assert NUM_FRAMES == 16, "Keep the original model sampling protocol."

if RUN_P1_PREFIX:
    p1_early_summary = []
    for frac in P1_PREFIX_FRACTIONS:
        if frac == 1.0:
            df_prefix = isolated_results.copy()
            df_prefix["original_t_end"] = df_prefix["t_end"].astype(int)
            df_prefix["prefix_end"] = df_prefix["t_end"].astype(int)
        else:
            partial = isolated_df.copy()
            dur = partial["t_end"].astype(int)-partial["t_start"].astype(int)+1
            original_end = partial[["sample_id", "t_end"]].rename(
                columns={"t_end": "original_t_end"})
            partial["t_end"] = np.minimum(
                partial["t_end"].astype(int),
                partial["t_start"].astype(int)+np.maximum(1, np.ceil(dur*frac).astype(int))-1)
            partial["duration_frames"] = partial["t_end"]-partial["t_start"]+1
            records = []
            for video_id, group in tqdm(
                list(partial.groupby("video_id", sort=False)), desc=f"CTP prefix {frac:.0%}"):
                recs, _, _ = run_isolated_video_sequential(
                    model, group, ISOLATED_BATCH_SIZE)
                records.extend(recs)
            df_prefix = pd.DataFrame(records).merge(
                original_end, on="sample_id", how="left", validate="one_to_one")
            df_prefix["prefix_end"] = df_prefix["t_end"].astype(int)
            df_prefix["t_end"] = df_prefix["original_t_end"].astype(int)
        df_prefix["observed_fraction"] = frac
        assert len(df_prefix)==len(isolated_df), "Incomplete prefix inference."
        df_prefix.to_csv(P1_CTP_DIR/f"isolated_prefix_{int(frac*100):03d}.csv", index=False)
        y = df_prefix["gt_idx"].to_numpy(dtype=int)
        yp = df_prefix["pred_idx"].to_numpy(dtype=int)
        gesture_mask = y != 0
        p1_early_summary.append({
            "observation_ratio": frac, "samples": int(len(y)),
            "accuracy_all_9_classes": float(accuracy_score(y, yp)),
            "macro_f1_all_9_classes": float(f1_score(
                y, yp, labels=list(range(NUM_CLASSES)),
                average="macro", zero_division=0)),
            "accuracy_gt_gesture_only": float(accuracy_score(y[gesture_mask], yp[gesture_mask])),
            "macro_f1_gt_gesture_only": float(f1_score(
                y[gesture_mask], yp[gesture_mask], labels=list(range(1,NUM_CLASSES)),
                average="macro", zero_division=0))
        })
    p1_early_table = pd.DataFrame(p1_early_summary).sort_values("observation_ratio")
    p1_early_table.to_csv(P1_CTP_DIR/"early_accuracy_curve.csv",index=False)
    x = p1_early_table.observation_ratio.to_numpy()
    auc = {}
    for key in ("accuracy_all_9_classes","accuracy_gt_gesture_only"):
        auc["auc_10_to_100_"+key] = float(
            np.sum((p1_early_table[key].to_numpy()[1:] + p1_early_table[key].to_numpy()[:-1]) * np.diff(x) / 2)/(x[-1]-x[0]))
    auc["note"] = "GT-defined isolated prefix; AUC normalized over [10%, 100%]."
    (P1_CTP_DIR/"early_auc.json").write_text(json.dumps(auc,indent=2))
    print("CTP early AUC:",auc)
    display(p1_early_table)


### P1 Isolated — clip-level contact sheets, error vs matched correct

Each row is ONE GT-annotated gesture clip sampled at 10/30/50/70/90%. Save error, correct (same GT class, nearest duration), and paired PNG figures. These are representative video frames, not learned keyframe relevance scores.


In [ ]:
# P1 isolated qualitative evaluation: pair wrong and correct cases from the SAME GT class.
# 5 evenly spaced representative frames per annotated clip; NOT learned attention keyframes.
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

P1_VIS_ROOT = P1_CTP_DIR
P1_ISO_DIR = P1_VIS_ROOT / "isolated_filmstrips"
P1_ISO_DIR.mkdir(parents=True, exist_ok=True)
P1_ISO_MAX_PAIRS = 6
P1_ISO_FRAME_RATIOS = np.array([.10, .30, .50, .70, .90], dtype=float)
P1_IS_IPN = False

p1_iso = isolated_results.copy()
if not P1_IS_IPN:
    # CTP isolated predictions do not carry video_path. Join by unique sample_id.
    lookup = test_segments[["sample_id", "video_path"]].drop_duplicates("sample_id")
    p1_iso = p1_iso.merge(lookup, on="sample_id", how="left", validate="one_to_one")
assert p1_iso["video_path"].notna().all(), "Missing video paths."
assert len(p1_iso) == len(isolated_results), "Incorrect isolated join."

def p1_iso_read_frames(case, ratios=P1_ISO_FRAME_RATIOS):
    path = Path(str(case["video_path"]))
    if not path.is_file():
        raise FileNotFoundError(path)
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot decode: {path}")
    start, end = int(case["t_start"]), int(case["t_end"])
    if end < start:
        cap.release()
        raise ValueError(f"Invalid annotation {start}..{end}")
    indices = np.rint(start + ratios * (end-start)).astype(int)
    frames = []
    try:
        for idx in indices:
            seek = int(idx) - (1 if P1_IS_IPN else 0)
            cap.set(cv2.CAP_PROP_POS_FRAMES, seek)
            ok, bgr = cap.read()
            if not ok:
                raise RuntimeError(f"Could not read frame {idx} of {path.name}")
            frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    finally:
        cap.release()
    return indices, frames

def p1_iso_render(rows, dest, title):
    fig, axes = plt.subplots(len(rows), 5, figsize=(16, 3.5*len(rows)),
                             squeeze=False, constrained_layout=True)
    for k, (case, status) in enumerate(rows):
        ids, frames = p1_iso_read_frames(case)
        gt, pred = int(case["gt_idx"]), int(case["pred_idx"])
        description = (
            f"{status}: GT={CLASS_CODES[gt]}, Pred={CLASS_CODES[pred]}\n"
            f"confidence={float(case['confidence']):.3f} | "
            f"frames {int(case['t_start'])}..{int(case['t_end'])}"
        )
        for j, (frame, idx) in enumerate(zip(frames, ids)):
            ax = axes[k, j]
            ax.imshow(frame)
            ax.axis("off")
            ax.set_title(f"{int(P1_ISO_FRAME_RATIOS[j]*100)}% | frame {idx}", fontsize=10)
            if j == 0:
                ax.set_ylabel(description, fontsize=9, rotation=0,
                              ha="right", va="center", labelpad=14)
    fig.suptitle(title + " | GT-annotated clip, representative frames only", fontsize=12)
    fig.savefig(dest, dpi=170, bbox_inches="tight")
    plt.close(fig)

p1_wrong = p1_iso[p1_iso.gt_idx != p1_iso.pred_idx].sort_values(
    "confidence", ascending=False)
p1_right = p1_iso[p1_iso.gt_idx == p1_iso.pred_idx].copy()
paired_meta = []
for rank, (_, wrong) in enumerate(p1_wrong.head(P1_ISO_MAX_PAIRS).iterrows(), start=1):
    possible = p1_right[p1_right.gt_idx == wrong.gt_idx]
    if len(possible):
        nearest = (possible["duration_frames"].astype(float) -
                   float(wrong["duration_frames"])).abs().sort_values().index[0]
        correct = possible.loc[nearest]
        p1_right = p1_right.drop(index=nearest)
    else:
        correct = None
    fail_file = P1_ISO_DIR / f"isolated_error_{rank:03d}.png"
    p1_iso_render([(wrong, "ERROR")], fail_file, f"Isolated error #{rank}")
    row = {"rank": rank, "gt": int(wrong.gt_idx), "error_pred": int(wrong.pred_idx),
           "error_video": str(wrong.video_path), "error_image": str(fail_file),
           "error_confidence": float(wrong.confidence),
           "correct_image": "", "paired_image": ""}
    if correct is not None:
        good_file = P1_ISO_DIR / f"isolated_correct_{rank:03d}.png"
        pair_file = P1_ISO_DIR / f"isolated_pair_{rank:03d}.png"
        p1_iso_render([(correct, "CORRECT")], good_file, f"Isolated correct #{rank}")
        p1_iso_render([(wrong, "ERROR"), (correct, "CORRECT")],
                      pair_file, f"Matched GT class {CLASS_CODES[int(wrong.gt_idx)]}")
        row.update({"correct_image": str(good_file),
                    "paired_image": str(pair_file),
                    "correct_video": str(correct.video_path),
                    "correct_confidence": float(correct.confidence)})
    paired_meta.append(row)
pd.DataFrame(paired_meta).to_csv(
    P1_ISO_DIR / "isolated_qualitative_pairs.csv", index=False)
print(f"Isolated: rendered {len(paired_meta)} misclassified clips; "
      f"pairs matched by GT class and nearest available duration. Saved to {P1_ISO_DIR}")


## 7. Continuous gesture recognition helpers

Continuous evaluation never uses ground-truth segment boundaries for model
input construction.

Only the full video is used to generate sliding windows.

For CTPGesture:

```text
WINDOW_SIZE = 32
WINDOW_STEP = 8
overlap     = 75%
MODEL_FRAMES = 16
offsets = [0,2,4,...,30]
```

Each window produces one 9-class probability vector. Every frame receives the
average probability of all windows covering it.

The raw frame prediction is then:

\[
\hat y_f = \arg\max_c P(c|f)
\]

Ground truth from the CSV is used only for scoring.

In [ ]:
import time
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

def get_video_info(video_path):
    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise ValueError(
            f"Cannot open video: {video_path}"
        )

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )
    fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )

    cap.release()

    if fps <= 0:
        fps = 15.0

    return total_frames, fps

def build_window_starts(
    total_frames,
    window_size,
    window_step,
):
    if total_frames <= window_size:
        return [0]

    last_full_start = (
        total_frames - window_size
    )

    starts = list(
        range(
            0,
            last_full_start + 1,
            window_step,
        )
    )

    if starts[-1] != last_full_start:
        starts.append(
            last_full_start
        )

    return starts

def build_sample_offsets(
    window_size,
    model_frames,
):
    assert (
        window_size % model_frames == 0
    )

    stride = (
        window_size // model_frames
    )

    offsets = np.arange(
        0,
        window_size,
        stride,
        dtype=np.int64,
    )

    assert len(offsets) == model_frames
    return offsets

def _build_window_target_indices(
    starts,
    sample_offsets,
    total_frames,
):
    targets = []

    for start in starts:
        idxs = np.clip(
            int(start) + sample_offsets,
            0,
            total_frames - 1,
        ).astype(np.int64)
        targets.append(idxs)

    return targets

def run_continuous_inference(
    model,
    video_id,
    show_progress=False,
    batch_size=CONTINUOUS_BATCH_SIZE,
):
    """
    Optimized continuous inference.

    Key properties:
    - MP4 is opened exactly once per video.
    - Frames are decoded sequentially.
    - No CAP_PROP_POS_FRAMES random seeks.
    - Overlapping windows reuse already-preprocessed frames.
    - Multiple windows are evaluated in one GPU forward pass.
    """
    video_path = (
        TEST_DIR / f"{video_id}.mp4"
    )
    csv_path = (
        TEST_DIR / f"{video_id}.csv"
    )

    gt_labels = read_ctp_labels(
        csv_path
    )

    total_frames_video, fps = (
        get_video_info(video_path)
    )

    usable_frames = min(
        total_frames_video,
        len(gt_labels),
    )

    gt_labels = gt_labels[
        :usable_frames
    ]
    total_frames = usable_frames

    starts = build_window_starts(
        total_frames,
        WINDOW_SIZE,
        WINDOW_STEP,
    )

    ends = [
        min(
            start + WINDOW_SIZE - 1,
            total_frames - 1,
        )
        for start in starts
    ]

    sample_offsets = (
        build_sample_offsets(
            WINDOW_SIZE,
            MODEL_FRAMES,
        )
    )

    window_targets = (
        _build_window_target_indices(
            starts,
            sample_offsets,
            total_frames,
        )
    )

    # Count how many future windows still need each frame.
    # This allows a small rolling frame cache even with overlap.
    from collections import Counter

    use_count = Counter()

    for target_idx in window_targets:
        use_count.update(
            target_idx.tolist()
        )

    needed_indices = set(
        use_count.keys()
    )

    frame_cache = {}

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise RuntimeError(
            f"Cannot open video: {video_path}"
        )

    current_frame_idx = 0

    p_raw = np.zeros(
        (
            len(starts),
            NUM_CLASSES,
        ),
        dtype=np.float32,
    )

    pending_clips = []
    pending_window_indices = []

    decode_preprocess_sec = 0.0
    model_forward_sec = 0.0

    wall_t0 = time.perf_counter()

    if show_progress:
        progress = tqdm(
            total=len(starts),
            desc=f"Continuous {video_id}",
        )
    else:
        progress = None

    def flush_batch():
        nonlocal pending_clips
        nonlocal pending_window_indices
        nonlocal model_forward_sec

        if not pending_clips:
            return

        probs_batch, forward_sec = (
            clips_to_probabilities(
                model,
                pending_clips,
                DEVICE,
            )
        )

        model_forward_sec += forward_sec

        for wi, probs in zip(
            pending_window_indices,
            probs_batch,
        ):
            p_raw[wi] = probs

        if progress is not None:
            progress.update(
                len(pending_clips)
            )

        pending_clips = []
        pending_window_indices = []

    for wi, target_idx in enumerate(
        window_targets
    ):
        max_target = int(
            target_idx.max()
        )

        t_decode = time.perf_counter()

        # Sequential decode only until this window's newest required frame.
        while (
            current_frame_idx
            <= max_target
        ):
            ok, frame_bgr = cap.read()

            if not ok:
                break

            if (
                current_frame_idx
                in needed_indices
            ):
                frame_rgb = cv2.cvtColor(
                    frame_bgr,
                    cv2.COLOR_BGR2RGB,
                )

                frame_cache[
                    current_frame_idx
                ] = preprocess_rgb_frame(
                    frame_rgb
                )

            current_frame_idx += 1

        decode_preprocess_sec += (
            time.perf_counter()
            - t_decode
        )

        missing = [
            int(idx)
            for idx in target_idx
            if int(idx)
            not in frame_cache
        ]

        if missing:
            cap.release()
            if progress is not None:
                progress.close()

            raise RuntimeError(
                f"{video_id}: missing decoded "
                f"sample frames {missing[:8]}"
            )

        frames = [
            frame_cache[
                int(idx)
            ]
            for idx in target_idx
        ]

        clip = (
            torch.stack(frames, dim=0)
            .permute(1, 0, 2, 3)
            .contiguous()
        )

        pending_clips.append(clip)
        pending_window_indices.append(wi)

        # This window has consumed its references. Drop frames as soon as
        # no future overlapping window needs them.
        for idx in target_idx.tolist():
            idx = int(idx)
            use_count[idx] -= 1

            if use_count[idx] <= 0:
                use_count.pop(idx, None)
                frame_cache.pop(
                    idx,
                    None,
                )
                needed_indices.discard(
                    idx
                )

        if (
            len(pending_clips)
            >= int(batch_size)
        ):
            flush_batch()

    flush_batch()

    if progress is not None:
        progress.close()

    cap.release()

    wall_elapsed = (
        time.perf_counter()
        - wall_t0
    )

    gt_segments = gesture_segments(
        gt_labels
    )

    return {
        "video_id": video_id,
        "video_path": str(video_path),
        "csv_path": str(csv_path),
        "fps": float(fps),
        "total_frames": int(total_frames),
        "duration_sec": float(
            total_frames / fps
        ),
        "window_starts": np.asarray(
            starts,
            dtype=np.int64,
        ),
        "window_ends": np.asarray(
            ends,
            dtype=np.int64,
        ),
        "sample_offsets": np.asarray(
            sample_offsets,
            dtype=np.int64,
        ),
        "num_windows": int(
            len(starts)
        ),
        "P_raw": p_raw,
        "gt_labels": gt_labels,
        "gt_segments": gt_segments,

        # New timing fields with unambiguous names.
        "wall_elapsed_sec": float(
            wall_elapsed
        ),
        "decode_preprocess_sec": float(
            decode_preprocess_sec
        ),
        "model_forward_sec": float(
            model_forward_sec
        ),
        "mean_wall_sec_per_window": (
            float(
                wall_elapsed
                / len(starts)
            )
            if starts
            else np.nan
        ),
        "mean_model_sec_per_window": (
            float(
                model_forward_sec
                / len(starts)
            )
            if starts
            else np.nan
        ),

        # Backward-compatible names used by later cells.
        "infer_elapsed_sec": float(
            wall_elapsed
        ),
        "mean_infer_sec_per_window": (
            float(
                wall_elapsed
                / len(starts)
            )
            if starts
            else np.nan
        ),
    }

def smooth_window_probs(
    window_probs,
    enabled=False,
):
    if (
        not enabled
        or len(window_probs) <= 1
    ):
        return np.asarray(
            window_probs,
            dtype=np.float32,
        ).copy()

    padded = np.pad(
        window_probs,
        ((1, 1), (0, 0)),
        mode="edge",
    )

    return (
        (
            padded[:-2]
            + padded[1:-1]
            + padded[2:]
        )
        / 3.0
    ).astype(np.float32)

def windows_to_frame_probs(
    window_probs,
    window_starts,
    window_ends,
    total_frames,
    num_classes=NUM_CLASSES,
):
    p_sum = np.zeros(
        (total_frames, num_classes),
        dtype=np.float64,
    )

    count = np.zeros(
        total_frames,
        dtype=np.float64,
    )

    for i, start in enumerate(
        window_starts
    ):
        end = int(
            window_ends[i]
        )

        p_sum[
            start:end + 1
        ] += window_probs[i]

        count[
            start:end + 1
        ] += 1.0

    uncovered = count == 0

    if np.any(uncovered):
        count[uncovered] = 1.0
        p_sum[
            uncovered,
            0,
        ] = 1.0

    return (
        p_sum / count[:, None]
    ).astype(np.float32)

def labels_to_segments(labels):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    if len(labels) == 0:
        return []

    segments = []
    start = 0
    current = int(labels[0])

    for i in range(1, len(labels)):
        value = int(labels[i])

        if value != current:
            segments.append({
                "label": current,
                "start": start,
                "end": i - 1,
            })
            start = i
            current = value

    segments.append({
        "label": current,
        "start": start,
        "end": len(labels) - 1,
    })

    return segments

def gesture_segments(labels):
    return [
        seg
        for seg in labels_to_segments(labels)
        if int(seg["label"]) != 0
    ]

def link_same_label_across_gaps(
    frame_labels,
    max_gap_frames,
    fps,
):
    linked = np.asarray(
        frame_labels,
        dtype=np.int64,
    ).copy()

    log = []

    while True:
        segs = labels_to_segments(
            linked
        )

        candidates = []

        for i, left in enumerate(segs):
            label = int(
                left["label"]
            )

            if label == 0:
                continue

            for j in range(
                i + 1,
                len(segs),
            ):
                right = segs[j]

                gap = int(
                    right["start"]
                    - left["end"]
                    - 1
                )

                if gap > max_gap_frames:
                    break

                if (
                    int(right["label"])
                    == label
                ):
                    if gap > 0:
                        candidates.append(
                            (
                                gap,
                                int(left["start"]),
                                i,
                                j,
                                label,
                            )
                        )
                    break

        if not candidates:
            break

        _, _, i, j, label = min(
            candidates,
            key=lambda x: (
                x[0],
                x[1],
            ),
        )

        segs = labels_to_segments(
            linked
        )

        left = segs[i]
        right = segs[j]

        gap_start = int(
            left["end"] + 1
        )
        gap_end = int(
            right["start"] - 1
        )

        old_labels = linked[
            gap_start:gap_end + 1
        ].copy()

        gap_frames = (
            gap_end
            - gap_start
            + 1
        )

        linked[
            gap_start:gap_end + 1
        ] = label

        log.append({
            "label": label,
            "gap_start": gap_start,
            "gap_end": gap_end,
            "gap_frames": gap_frames,
            "gap_sec": gap_frames / fps,
            "replaced_labels": sorted(
                np.unique(old_labels)
                .astype(int)
                .tolist()
            ),
        })

    return linked, log

def remove_short_gesture_segments(
    frame_labels,
    min_frames,
    fps,
    fill_same_neighbors=True,
):
    src = np.asarray(
        frame_labels,
        dtype=np.int64,
    )

    out = src.copy()
    segs = labels_to_segments(src)
    log = []

    for i, seg in enumerate(segs):
        label = int(
            seg["label"]
        )

        if label == 0:
            continue

        n_frames = int(
            seg["end"]
            - seg["start"]
            + 1
        )

        if n_frames >= min_frames:
            continue

        new_label = 0
        reason = "short_to_background"

        if (
            fill_same_neighbors
            and 0 < i < len(segs) - 1
        ):
            prev_seg = segs[i - 1]
            next_seg = segs[i + 1]

            prev_len = int(
                prev_seg["end"]
                - prev_seg["start"]
                + 1
            )

            next_len = int(
                next_seg["end"]
                - next_seg["start"]
                + 1
            )

            if (
                int(prev_seg["label"]) != 0
                and int(prev_seg["label"])
                == int(next_seg["label"])
                and prev_len >= min_frames
                and next_len >= min_frames
            ):
                new_label = int(
                    prev_seg["label"]
                )
                reason = (
                    "short_between_same_label"
                )

        out[
            int(seg["start"]):
            int(seg["end"]) + 1
        ] = new_label

        log.append({
            "start_frame": int(seg["start"]),
            "end_frame": int(seg["end"]),
            "duration_frames": n_frames,
            "duration_sec": n_frames / fps,
            "old_label": label,
            "new_label": new_label,
            "reason": reason,
        })

    return out, log

def temporal_iou(a, b):
    inter = max(
        min(a["end"], b["end"])
        - max(a["start"], b["start"])
        + 1,
        0,
    )

    union = (
        max(a["end"], b["end"])
        - min(a["start"], b["start"])
        + 1
    )

    return (
        0.0
        if union <= 0
        else inter / union
    )

def overlap_score(
    gt,
    pred,
    theta_frames,
):
    used = set()
    scores = []

    gt = sorted(
        gt,
        key=lambda x: x["start"],
    )
    pred = sorted(
        pred,
        key=lambda x: x["start"],
    )

    for g in gt:
        candidates = []

        for j, p in enumerate(pred):
            if (
                j in used
                or p["label"] != g["label"]
            ):
                continue

            if (
                abs(
                    p["start"] - g["start"]
                ) > theta_frames
            ):
                continue

            if (
                abs(
                    p["end"] - g["end"]
                ) > theta_frames
            ):
                continue

            candidates.append(
                (
                    temporal_iou(p, g),
                    j,
                )
            )

        if candidates:
            score, j = max(
                candidates,
                key=lambda x: x[0],
            )

            used.add(j)
            scores.append(score)
        else:
            scores.append(0.0)

    unmatched_pred = (
        len(pred) - len(used)
    )

    denom = (
        len(gt) + unmatched_pred
    )

    return (
        1.0
        if denom == 0
        else float(
            np.sum(scores) / denom
        )
    )

def gesture_recall_with_tolerance(
    gt,
    pred,
    theta_frames,
):
    gt = sorted(
        gt,
        key=lambda x: x["start"],
    )
    pred = sorted(
        pred,
        key=lambda x: x["start"],
    )

    if len(gt) == 0:
        return 1.0

    used = set()
    matched = 0

    for g in gt:
        candidates = []

        for j, p in enumerate(pred):
            if (
                j in used
                or int(p["label"])
                != int(g["label"])
            ):
                continue

            if (
                abs(
                    int(p["start"])
                    - int(g["start"])
                ) > theta_frames
            ):
                continue

            if (
                abs(
                    int(p["end"])
                    - int(g["end"])
                ) > theta_frames
            ):
                continue

            candidates.append(
                (
                    temporal_iou(p, g),
                    j,
                )
            )

        if candidates:
            _, best_j = max(
                candidates,
                key=lambda x: x[0],
            )
            used.add(best_j)
            matched += 1

    return matched / len(gt)

def frame_metric_dict(
    y_true,
    y_pred,
):
    return {
        "frame_accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_precision": float(
            precision_score(
                y_true,
                y_pred,
                labels=list(range(NUM_CLASSES)),
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true,
                y_pred,
                labels=list(range(NUM_CLASSES)),
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(range(NUM_CLASSES)),
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(range(NUM_CLASSES)),
                average="weighted",
                zero_division=0,
            )
        ),
    }

def apply_postprocess_to_cache(
    cache_entry,
    *,
    enable_smoothing=False,
    t_avg_sec=0.0,
    min_gesture_sec=0.0,
    fill_same_neighbors=False,
    theta_sec=THETA_SEC,
):
    fps = float(
        cache_entry["fps"]
    )

    total_frames = int(
        cache_entry["total_frames"]
    )

    starts = np.asarray(
        cache_entry["window_starts"],
        dtype=np.int64,
    )

    ends = np.asarray(
        cache_entry["window_ends"],
        dtype=np.int64,
    )

    p_raw = np.asarray(
        cache_entry["P_raw"],
        dtype=np.float32,
    )

    gt_labels = np.asarray(
        cache_entry["gt_labels"],
        dtype=np.int64,
    )

    gt_segments = [
        dict(seg)
        for seg in cache_entry["gt_segments"]
    ]

    p_stage = smooth_window_probs(
        p_raw,
        enabled=enable_smoothing,
    )

    p_frame = windows_to_frame_probs(
        p_stage,
        starts,
        ends,
        total_frames,
    )

    stage_labels = np.argmax(
        p_frame,
        axis=1,
    ).astype(np.int64)

    max_gap_frames = int(
        round(
            float(t_avg_sec) * fps
        )
    )

    min_frames = int(
        round(
            float(min_gesture_sec) * fps
        )
    )

    theta_frames = int(
        round(
            float(theta_sec) * fps
        )
    )

    linked_labels, linking_log = (
        link_same_label_across_gaps(
            stage_labels,
            max_gap_frames,
            fps,
        )
    )

    final_labels, removal_log = (
        remove_short_gesture_segments(
            linked_labels,
            min_frames,
            fps,
            fill_same_neighbors=(
                fill_same_neighbors
            ),
        )
    )

    pred_segments = gesture_segments(
        final_labels
    )

    metrics = frame_metric_dict(
        gt_labels,
        final_labels,
    )

    metrics.update({
        "temporal_overlap": overlap_score(
            gt_segments,
            pred_segments,
            theta_frames,
        ),
        "gesture_recall": (
            gesture_recall_with_tolerance(
                gt_segments,
                pred_segments,
                theta_frames,
            )
        ),
        "pred_segments": int(
            len(pred_segments)
        ),
        "gt_segments": int(
            len(gt_segments)
        ),
        "link_operations": int(
            len(linking_log)
        ),
        "removal_changes": int(
            len(removal_log)
        ),
        "correct_frames": int(
            np.sum(
                final_labels
                == gt_labels
            )
        ),
        "total_frames": int(
            len(gt_labels)
        ),
    })

    return {
        "metrics": metrics,
        "final_labels": final_labels,
        "stage_labels": stage_labels,
        "p_frame": p_frame,
        "pred_segments": pred_segments,
        "linking_log": linking_log,
        "removal_log": removal_log,
    }

print("[OK] Continuous helpers ready.")
print(
    "Sample offsets:",
    build_sample_offsets(
        WINDOW_SIZE,
        MODEL_FRAMES,
    ).tolist(),
)

### 7.2 Full official continuous test split

This cell runs VideoMamba inference **once** over the official 11 test videos and
stores the raw window probabilities.

The resulting cache is reused by the post-processing ablation section, so the
model is not re-run for every threshold configuration.

In [ ]:
import pickle
from tqdm.auto import tqdm
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
)

debug_cache = None  # P1 standalone: skip separate debug inference pass
continuous_raw_cache = {}
continuous_raw_rows = []
continuous_raw_summary = None

if RUN_CONTINUOUS:
    selected_ids = TEST_IDS.copy()

    if CONTINUOUS_MAX_VIDEOS is not None:
        selected_ids = selected_ids[
            :int(CONTINUOUS_MAX_VIDEOS)
        ]

    all_gt = []
    all_pred = []

    total_correct = 0
    total_frames = 0
    total_windows = 0

    total_wall_sec = 0.0
    total_decode_preprocess_sec = 0.0
    total_model_forward_sec = 0.0

    for video_id in tqdm(
        selected_ids,
        desc="Official continuous videos",
    ):
        cache_path = (
            PER_VIDEO_CACHE_DIR
            / f"{video_id}_raw.pkl"
        )

        cache_entry = None
        cache_source = "fresh"

        # Reuse the optional debug result rather than running video 002 twice.
        if (
            REUSE_DEBUG_CACHE
            and debug_cache is not None
            and video_id
            == debug_cache.get(
                "video_id"
            )
        ):
            cache_entry = debug_cache
            cache_source = "debug-cache"

        elif (
            REUSE_PER_VIDEO_CACHE
            and cache_path.is_file()
        ):
            with cache_path.open(
                "rb"
            ) as f:
                cache_entry = (
                    pickle.load(f)
                )
            cache_source = "disk-cache"

        if cache_entry is None:
            cache_entry = (
                run_continuous_inference(
                    model,
                    video_id,
                    show_progress=False,
                    batch_size=(
                        CONTINUOUS_BATCH_SIZE
                    ),
                )
            )

            if SAVE_PER_VIDEO_CACHE:
                with cache_path.open(
                    "wb"
                ) as f:
                    pickle.dump(
                        cache_entry,
                        f,
                        protocol=(
                            pickle.HIGHEST_PROTOCOL
                        ),
                    )

        continuous_raw_cache[
            video_id
        ] = cache_entry

        raw_result = (
            apply_postprocess_to_cache(
                cache_entry,
                enable_smoothing=False,
                t_avg_sec=0.0,
                min_gesture_sec=0.0,
                fill_same_neighbors=False,
                theta_sec=THETA_SEC,
            )
        )

        metrics = raw_result[
            "metrics"
        ]

        continuous_raw_rows.append({
            "video_id": video_id,
            "cache_source": cache_source,
            "frames": cache_entry[
                "total_frames"
            ],
            "fps": cache_entry[
                "fps"
            ],
            "duration_sec": cache_entry[
                "duration_sec"
            ],
            "windows": cache_entry[
                "num_windows"
            ],
            "wall_sec": cache_entry[
                "wall_elapsed_sec"
            ],
            "decode_preprocess_sec": (
                cache_entry[
                    "decode_preprocess_sec"
                ]
            ),
            "model_forward_sec": (
                cache_entry[
                    "model_forward_sec"
                ]
            ),
            "wall_ms_per_window": (
                1000
                * cache_entry[
                    "mean_wall_sec_per_window"
                ]
            ),
            "model_ms_per_window": (
                1000
                * cache_entry[
                    "mean_model_sec_per_window"
                ]
            ),
            **metrics,
        })

        gt = np.asarray(
            cache_entry["gt_labels"],
            dtype=np.int64,
        )

        pred = np.asarray(
            raw_result["final_labels"],
            dtype=np.int64,
        )

        all_gt.append(gt)
        all_pred.append(pred)

        total_correct += int(
            metrics["correct_frames"]
        )
        total_frames += int(
            metrics["total_frames"]
        )
        total_windows += int(
            cache_entry["num_windows"]
        )

        # Only fresh work should contribute to current-run execution-time
        # totals. Cached entries retain their original timings and are still
        # included below as protocol/runtime observations.
        total_wall_sec += float(
            cache_entry["wall_elapsed_sec"]
        )
        total_decode_preprocess_sec += float(
            cache_entry[
                "decode_preprocess_sec"
            ]
        )
        total_model_forward_sec += float(
            cache_entry[
                "model_forward_sec"
            ]
        )

        print(
            f"[{video_id}] "
            f"source={cache_source} | "
            f"windows={cache_entry['num_windows']} | "
            f"wall={cache_entry['wall_elapsed_sec']:.1f}s | "
            f"decode={cache_entry['decode_preprocess_sec']:.1f}s | "
            f"model={cache_entry['model_forward_sec']:.1f}s"
        )

    continuous_raw_df = pd.DataFrame(
        continuous_raw_rows
    )

    y_true_global = np.concatenate(
        all_gt
    )
    y_pred_global = np.concatenate(
        all_pred
    )

    global_metrics = frame_metric_dict(
        y_true_global,
        y_pred_global,
    )

    continuous_raw_summary = {
        "videos": int(
            len(continuous_raw_df)
        ),
        "frames": int(
            total_frames
        ),
        "windows": int(
            total_windows
        ),
        "mean_per_video_frame_accuracy": float(
            continuous_raw_df[
                "frame_accuracy"
            ].mean()
        ),
        "global_frame_accuracy": float(
            total_correct
            / total_frames
        ),
        "global_balanced_accuracy": float(
            global_metrics[
                "balanced_accuracy"
            ]
        ),
        "global_macro_precision": float(
            global_metrics[
                "macro_precision"
            ]
        ),
        "global_macro_recall": float(
            global_metrics[
                "macro_recall"
            ]
        ),
        "global_macro_f1": float(
            global_metrics[
                "macro_f1"
            ]
        ),
        "global_weighted_f1": float(
            global_metrics[
                "weighted_f1"
            ]
        ),
        "mean_temporal_overlap": float(
            continuous_raw_df[
                "temporal_overlap"
            ].mean()
        ),
        "mean_gesture_recall": float(
            continuous_raw_df[
                "gesture_recall"
            ].mean()
        ),
        "total_wall_sec": float(
            total_wall_sec
        ),
        "total_decode_preprocess_sec": float(
            total_decode_preprocess_sec
        ),
        "total_model_forward_sec": float(
            total_model_forward_sec
        ),
        "mean_wall_ms_per_window": float(
            1000
            * total_wall_sec
            / total_windows
        ),
        "mean_model_ms_per_window": float(
            1000
            * total_model_forward_sec
            / total_windows
        ),
        "effective_windows_per_second": float(
            total_windows
            / total_wall_sec
        ),
    }

    print("=" * 72)
    print(
        "CONTINUOUS RAW — "
        "OFFICIAL TEST SPLIT"
    )
    print("=" * 72)

    for key, value in (
        continuous_raw_summary.items()
    ):
        if isinstance(
            value,
            (float, np.floating),
        ):
            print(
                f"{key:32s}: "
                f"{value:.6f}"
            )
        else:
            print(
                f"{key:32s}: "
                f"{value}"
            )

    display(continuous_raw_df)

    continuous_raw_df.to_csv(
        OUTPUT_DIR
        / "continuous_raw_per_video.csv",
        index=False,
    )

    combined_cache_path = (
        OUTPUT_DIR
        / "continuous_raw_cache.pkl"
    )

    with combined_cache_path.open(
        "wb"
    ) as f:
        pickle.dump(
            continuous_raw_cache,
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    print(
        "Saved combined cache:",
        combined_cache_path,
    )

    cm = confusion_matrix(
        y_true_global,
        y_pred_global,
        labels=list(
            range(NUM_CLASSES)
        ),
        normalize="true",
    )

    fig, ax = plt.subplots(
        figsize=(10, 8)
    )

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_CODES,
    ).plot(
        ax=ax,
        values_format=".2f",
        colorbar=True,
    )

    ax.set_title(
        "CTPGesture continuous RAW — "
        "normalized confusion matrix"
    )
    plt.tight_layout()
    plt.show()

else:
    print("[SKIP] RUN_CONTINUOUS=False")

### 7.4 P1: RAW continuous F1@10/25/50, Edit and qualitative case manifests

Temporal IoU segment metrics and edit score follow MS-TCN conventions. Export all isolated/continuous errors and correct examples for choosing figure layouts. ECE/Brier/NLL uses original isolated softmax without test calibration.


In [ ]:
# P1 continuous RAW temporal segmentation metrics and visualization case manifests.
# Frame metrics vs segment metrics are NOT equivalent to causal online recognition.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

P1_CTP_DIR = OUTPUT_DIR / "p1_metric_audit"
P1_CTP_DIR.mkdir(parents=True, exist_ok=True)

def p1_segments(labels, ignore_background=True):
    a = np.asarray(labels,dtype=np.int64)
    if not len(a): return []
    breaks = np.r_[0,np.flatnonzero(a[1:]!=a[:-1])+1,len(a)]
    return [(int(a[s]),int(s),int(e-1)) for s,e in zip(breaks[:-1],breaks[1:])
            if not(ignore_background and int(a[s])==0)]

def p1_edit_distance(a,b):
    previous=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        current=[i]+[0]*len(b)
        for j,y in enumerate(b,1):
            current[j]=min(current[j-1]+1,previous[j]+1,previous[j-1]+(x!=y))
        previous=current
    return int(previous[-1])

def p1_seg_counts(gt,pred,thresh):
    used=set();tp=fp=0
    for label,ps,pe in pred:
        best=None;best_iou=-1.
        for i,(glabel,gs,ge) in enumerate(gt):
            if i in used or label!=glabel: continue
            inter=max(0,min(pe,ge)-max(ps,gs)+1)
            union=(pe-ps+1)+(ge-gs+1)-inter
            iou=inter/union if union else 0.
            if iou>best_iou: best=i;best_iou=iou
        if best is not None and best_iou>=thresh:
            tp+=1;used.add(best)
        else: fp+=1
    return np.array([tp,fp,len(gt)-tp],dtype=int)

def p1_ece(probs,truth,bins=15):
    confidence=probs.max(axis=1);ok=(probs.argmax(axis=1)==truth)
    breaks=np.linspace(0.,1.,bins+1)
    total=0.
    for k in range(bins):
        m=(confidence>=breaks[k]) & ((confidence<breaks[k+1]) if k<bins-1 else (confidence<=breaks[k+1]))
        if m.any(): total+=float(m.mean()*abs(ok[m].mean()-confidence[m].mean()))
    return total

iso = isolated_results.copy()
cols = [f"prob_{j}" for j in range(NUM_CLASSES)]
assert set(cols).issubset(iso.columns)
probs=iso[cols].to_numpy(dtype=float)
truth=iso.gt_idx.to_numpy(dtype=int)
assert np.allclose(probs.sum(axis=1),1.,atol=1e-4)
calibration={
    "n":int(len(iso)),
    "ece_15":p1_ece(probs,truth),
    "multiclass_brier_sum":float(np.sum((probs-np.eye(NUM_CLASSES)[truth])**2,axis=1).mean()),
    "nll":float(-np.log(np.clip(probs[np.arange(len(truth)),truth],1e-12,1.)).mean())
}
(P1_CTP_DIR/"isolated_calibration.json").write_text(json.dumps(calibration,indent=2))
iso[iso.gt_idx!=iso.pred_idx].sort_values("confidence",ascending=False).to_csv(
    P1_CTP_DIR/"isolated_error_cases.csv",index=False)
iso[iso.gt_idx==iso.pred_idx].sort_values("confidence",ascending=False).to_csv(
    P1_CTP_DIR/"isolated_correct_cases.csv",index=False)

assert continuous_raw_cache, "Run full official continuous Section 7.2 first."
per_video=[];all_gt=[];all_pred=[]
counts={t:np.zeros(3,dtype=int) for t in (0.10,0.25,0.50)}
errors=[];controls=[]
for video_id,cache in sorted(continuous_raw_cache.items()):
    result=apply_postprocess_to_cache(
        cache,enable_smoothing=False,t_avg_sec=0.,min_gesture_sec=0.,
        fill_same_neighbors=False,theta_sec=THETA_SEC)
    gt=np.asarray(cache["gt_labels"],dtype=int)
    pred=np.asarray(result["stage_labels"],dtype=int)
    assert len(gt)==len(pred) and len(gt)>0
    gs,ps=p1_segments(gt),p1_segments(pred)
    row={"video_id":str(video_id),"frames":len(gt),
         "frame_accuracy":float(accuracy_score(gt,pred)),
         "macro_f1":float(f1_score(gt,pred,labels=list(range(NUM_CLASSES)),
                                     average="macro",zero_division=0))}
    for t in counts:
        cnt=p1_seg_counts(gs,ps,t);counts[t]+=cnt
        row[f"segment_f1_at_{int(t*100)}"]=float(2*cnt[0]/(2*cnt[0]+cnt[1]+cnt[2])) if (2*cnt[0]+cnt[1]+cnt[2]) else 0.
    dist=p1_edit_distance([s[0] for s in gs],[s[0] for s in ps])
    row["edit_distance"]=dist
    row["edit_score_ms_tcn_pct"]=100*(1-dist/max(len(gs),len(ps),1))
    per_video.append(row)
    all_gt.append(gt);all_pred.append(pred)
    for correct,destination in ((False,errors),(True,controls)):
        matches=(gt==pred) if correct else (gt!=pred)
        if correct: matches &= (gt!=0)
        for _,start,end in p1_segments(matches.astype(int)):
            middle=(start+end)//2
            destination.append({
                "video_id":str(video_id),"video_path":str(TEST_DIR/f"{video_id}.mp4"),
                "start":start,"end":end,"indexing":"CTP_0_based",
                "gt_idx":int(gt[middle]),"pred_idx":int(pred[middle]),
                "duration_frames":end-start+1,"correct":bool(correct)})
per=pd.DataFrame(per_video)
per.to_csv(P1_CTP_DIR/"continuous_raw_segment_metrics_per_video.csv",index=False)
for fname,cases in (("continuous_error_cases.csv",errors),("continuous_correct_cases.csv",controls)):
    d=pd.DataFrame(cases).sort_values("duration_frames",ascending=False) if cases else pd.DataFrame()
    d.to_csv(P1_CTP_DIR/fname,index=False)
global_gt=np.concatenate(all_gt);global_pred=np.concatenate(all_pred)
summary={
    "videos":int(len(per)),"frames":int(len(global_gt)),
    "global_frame_accuracy":float(accuracy_score(global_gt,global_pred)),
    "global_macro_f1":float(f1_score(global_gt,global_pred,
        labels=list(range(NUM_CLASSES)),average="macro",zero_division=0)),
    "mean_edit_score_ms_tcn_pct":float(per.edit_score_ms_tcn_pct.mean()),
    "protocol":"RAW offline 32-frame/8-stride; background class 0 excluded from segment matching"}
for t,(tp,fp,fn) in counts.items():
    summary[f"segment_f1_at_{int(t*100)}"]=float(2*tp/(2*tp+fp+fn)) if 2*tp+fp+fn else 0.
(P1_CTP_DIR/"continuous_raw_segment_metrics_summary.json").write_text(json.dumps(summary,indent=2))
print("CTP isolated calibration:",calibration)
print("CTP continuous RAW:",summary)
print("Saved error/control CSV manifests for a subsequent image-visualization cell.")


### P1 Continuous — keyframes with temporal context (correct vs error)

For each complete GT gesture segment, compare predicted class and temporal IoU. Display five representative video frames plus time-aligned GT/RAW Prediction stripes. False positives are also eligible. The visualization is *offline*, not causal. Output PNG + candidate CSV.


In [ ]:
# P1 continuous qualitative evaluation: temporal segments, not isolated clips.
# Error examples: missed gesture / class confusion / boundary mismatch / false positive.
# Correct controls: GT gesture segment with class-matched predicted segment tIoU >= 0.50.
# Each case has 5 original frames and a CONTEXTUAL GT-vs-RAW-Pred timeline.
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

P1_CONT_DIR = P1_CTP_DIR / "continuous_timeline_keyframes"
P1_CONT_DIR.mkdir(parents=True, exist_ok=True)
P1_CONT_MAX_ERRORS = 8
P1_CONT_MAX_CORRECT = 8
P1_CONT_CONTEXT_SEC = 1.5

def p1_cont_iou(a, b):
    # Each segment = (label, start, end), inclusive, 0-based.
    inter = max(0, min(a[2],b[2]) - max(a[1],b[1]) + 1)
    union = (a[2]-a[1]+1) + (b[2]-b[1]+1) - inter
    return inter / union if union else 0.

def p1_cont_render(case, gt, pred, fps, path, dest):
    start, end = int(case["start"]), int(case["end"])
    n = len(gt)
    if not Path(path).is_file():
        raise FileNotFoundError(path)
    selected = np.rint(np.linspace(start, end, 5)).astype(int)
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video {path}")
    frames = []
    try:
        for idx in selected:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, bgr = cap.read()
            if not ok:
                raise RuntimeError(f"Cannot read frame {idx} of {path}")
            frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    finally:
        cap.release()

    fig = plt.figure(figsize=(16, 5.4), constrained_layout=True)
    grid = fig.add_gridspec(2, 5, height_ratios=[2.3, 1.0])
    for j, (img, idx) in enumerate(zip(frames, selected)):
        ax = fig.add_subplot(grid[0, j])
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"frame {idx} | {idx/fps:.2f}s", fontsize=10)
    ax = fig.add_subplot(grid[1, :])
    context = max(10, int(np.ceil(P1_CONT_CONTEXT_SEC * fps)))
    left = max(0, start-context)
    right = min(n-1, end+context)
    cmap = ListedColormap(plt.get_cmap("tab20", NUM_CLASSES)(
        np.arange(NUM_CLASSES)))
    ax.imshow(np.vstack([gt[left:right+1], pred[left:right+1]]),
              interpolation="nearest", aspect="auto", cmap=cmap,
              vmin=-0.5, vmax=NUM_CLASSES-0.5,
              extent=(left-0.5,right+0.5,1.5,-0.5))
    ax.axvspan(start-0.5, end+0.5,
               color=("red" if case["type"]=="error" else "green"),
               alpha=0.15)
    ax.axvline(start-0.5, color="black", linewidth=0.8, linestyle="--")
    ax.axvline(end+0.5, color="black", linewidth=0.8, linestyle="--")
    ax.set_yticks([0,1],["Ground Truth","RAW Prediction"])
    ax.set_xlim(left-0.5,right+0.5)
    ax.set_xlabel("Frame index (0-based); prediction is OFFLINE, may use future frames")
    ax.grid(axis="x", alpha=0.2)
    fig.suptitle(
        f"CONTINUOUS {case['type'].upper()} | video {case['video']} | "
        f"{case['reason']} | GT={CLASS_CODES[int(case['gt_class'])]} "
        f"Pred(mid)={CLASS_CODES[int(case['pred_class'])]} | "
        f"segment={start}..{end}, tIoU={case['best_iou']:.2f}", fontsize=11)
    fig.savefig(dest, dpi=170, bbox_inches="tight")
    plt.close(fig)

p1_video_arrays = {}
p1_cases = []
for video, entry in sorted(continuous_raw_cache.items()):
    gt = np.asarray(entry["gt_labels"], dtype=np.int64)
    raw = windows_to_frame_probs(
        np.asarray(entry["P_raw"]), np.asarray(entry["window_starts"]),
        np.asarray(entry["window_ends"]), len(gt))
    pred = raw.argmax(axis=1).astype(np.int64)
    fps = max(1., float(entry["fps"]))
    path = str(entry.get("video_path", TEST_DIR / f"{video}.mp4"))
    p1_video_arrays[str(video)] = (gt,pred,fps,path)
    g_segments = p1_segments(gt)
    p_segments = p1_segments(pred)
    # Evaluate complete annotated gesture intervals, not merely consecutive correct frames.
    for g in g_segments:
        same = [p for p in p_segments if p[0]==g[0]]
        best = max([p1_cont_iou(g,p) for p in same], default=0.)
        start,end = g[1],g[2]
        mid = (start+end)//2
        pmid = int(pred[mid])
        if best >= 0.50:
            typ, reason = "correct", "correct_class_tIoU_ge_0.50"
        elif best > 0.:
            typ, reason = "error", "boundary_or_fragmentation"
        elif np.any(pred[start:end+1] != 0):
            typ, reason = "error", "class_confusion"
        else:
            typ, reason = "error", "missed_gesture"
        p1_cases.append({
            "video":str(video),"type":typ,"reason":reason,
            "start":int(start),"end":int(end),"duration_frames":int(end-start+1),
            "gt_class":int(g[0]),"pred_class":pmid,"best_iou":float(best),
            "fps":fps,"video_path":str(path)})
    # Explicit false-positive proposals: most GT frames are background.
    for p in p_segments:
        same = [g for g in g_segments if g[0]==p[0]]
        best = max([p1_cont_iou(p,g) for g in same],default=0.)
        st,en=p[1],p[2]
        if best < 0.10 and np.mean(gt[st:en+1]==0)>0.5:
            mid=(st+en)//2
            p1_cases.append({
                "video":str(video),"type":"error","reason":"false_positive_on_background",
                "start":int(st),"end":int(en),"duration_frames":int(en-st+1),
                "gt_class":int(gt[mid]),"pred_class":int(pred[mid]),
                "best_iou":float(best),"fps":fps,"video_path":str(path)})

case_df = pd.DataFrame(p1_cases)
case_df.to_csv(P1_CONT_DIR / "continuous_segment_candidates.csv", index=False)
if not case_df.empty:
    errors=case_df[case_df.type=="error"].sort_values("duration_frames",ascending=False)
    good=case_df[case_df.type=="correct"].sort_values("duration_frames",ascending=False)
    # Select diverse error types before filling remaining slots by duration.
    error_list=[];seen=set()
    for _, candidate in errors.iterrows():
        if candidate["reason"] not in seen:
            error_list.append(candidate); seen.add(candidate["reason"])
    selected_starts={(str(x["video"]),int(x["start"]),int(x["end"])) for x in error_list}
    for _, candidate in errors.iterrows():
        key=(str(candidate["video"]),int(candidate["start"]),int(candidate["end"]))
        if len(error_list)>=P1_CONT_MAX_ERRORS: break
        if key not in selected_starts:
            error_list.append(candidate); selected_starts.add(key)
    error_list=error_list[:P1_CONT_MAX_ERRORS]
    for typ, group in (("error",error_list),
                       ("correct",[row for _,row in good.head(P1_CONT_MAX_CORRECT).iterrows()])):
        for rank, case in enumerate(group,1):
            gt,pred,fps,path=p1_video_arrays[str(case["video"])]
            outfile=P1_CONT_DIR/f"continuous_{typ}_{rank:03d}.png"
            p1_cont_render(case,gt,pred,fps,path,outfile)
    print(f"Continuous: {len(errors)} error candidates, {len(good)} correct "
          f"gesture segments; rendered {len(error_list)} errors and "
          f"{min(len(good),P1_CONT_MAX_CORRECT)} correct controls in {P1_CONT_DIR}")
else:
    print("No continuous gesture segments to visualize.")
